<a href="https://colab.research.google.com/github/AliAI11/DolphinMind/blob/main/notebooks/04_final_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [20]:
!pip install -q torch transformers bitsandbytes accelerate rouge-score sentence-transformers faiss-cpu

print("all dependencies installed")

all dependencies installed


In [21]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from rouge_score import rouge_scorer
import time
import psutil
import json
import numpy as np

print("imports successful")

imports successful


In [22]:
import urllib.request

url = "https://www.gutenberg.org/files/1727/1727-0.txt"

with urllib.request.urlopen(url) as response:
    long_context = response.read().decode('utf-8')

# clean up gutenberg header/footer
start_marker = "*** START OF"
end_marker = "*** END OF"

if start_marker in long_context:
    long_context = long_context.split(start_marker)[1]
if end_marker in long_context:
    long_context = long_context.split(end_marker)[0]

# make it 2x longer for testing
long_context = long_context * 2

# test queries about the odyssey
test_queries = [
    "What is the Telemachy?",
    "Who is Polyphemos?",
    "What happened to Odysseus during his wanderings?"
]

reference_answers = [
    "The Telemachy is the first four books of the Odyssey focusing on Telemachos.",
    "Polyphemos is a Cyclops, son of Poseidon, who was blinded by Odysseus.",
    "Odysseus wandered for years after the Trojan War facing various challenges."
]

print(f"loaded document: {len(long_context.split()):,} words")
print(f"test queries: {len(test_queries)}")


loaded document: 259,156 words
test queries: 3


In [23]:
print("\n" + "="*60)
print("loading smollm3-3b with yarn for extended context")
print("="*60 + "\n")

from transformers import AutoConfig

model_name_smol = "HuggingFaceTB/SmolLM3-3B"

# load and modify config BEFORE loading model
config = AutoConfig.from_pretrained(model_name_smol, trust_remote_code=True)

# configure yarn for 128k context (must be done before model load)
config.rope_scaling = {
    "factor": 2.0,  # 2x65536 = 131,072 tokens
    "original_max_position_embeddings": 65536,
    "type": "yarn"
}
config.max_position_embeddings = 131072

print(f"configured yarn scaling: {config.rope_scaling}")
print(f"max context: {config.max_position_embeddings:,} tokens")

# 4-bit quantization
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

print(f"\nloading {model_name_smol} with modified config...")

# load model with yarn-enabled config
model_smol = AutoModelForCausalLM.from_pretrained(
    model_name_smol,
    config=config,  # critical: pass modified config
    quantization_config=quant_config,
    device_map="auto",
    trust_remote_code=True
)

tokenizer_smol = AutoTokenizer.from_pretrained(model_name_smol, trust_remote_code=True)

print(f"smollm3 loaded successfully")
print(f"configured max context: {model_smol.config.max_position_embeddings:,} tokens")
print(f"ram usage: {psutil.virtual_memory().used / 1e9:.2f} gb\n")


loading smollm3-3b with yarn for extended context

configured yarn scaling: {'factor': 2.0, 'original_max_position_embeddings': 65536, 'type': 'yarn'}
max context: 131,072 tokens

loading HuggingFaceTB/SmolLM3-3B with modified config...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

smollm3 loaded successfully
configured max context: 131,072 tokens
ram usage: 5.97 gb



In [24]:
def ask_smollm(context, query, max_context_tokens=4000):
    """generate response using smollm3 with proper context truncation"""

    # truncate context to specified length (prevents tokenizer warning)
    context_tokens = tokenizer_smol.encode(
        context,
        add_special_tokens=False,
        truncation=True,
        max_length=max_context_tokens
    )
    truncated_context = tokenizer_smol.decode(context_tokens, skip_special_tokens=True)

    # build messages using chat format
    messages = [
        {"role": "system", "content": "/no_think"},
        {"role": "user", "content": f"Context: {truncated_context}\n\nQuestion: {query}\n\nAnswer:"}
    ]

    # apply chat template
    prompt = tokenizer_smol.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    # tokenize
    inputs = tokenizer_smol(
        prompt,
        return_tensors="pt",
        truncation=False
    ).to(model_smol.device)

    actual_tokens = inputs['input_ids'].shape[1]

    # generate with recommended parameters
    outputs = model_smol.generate(
        **inputs,
        max_new_tokens=150,
        temperature=0.6,
        top_p=0.95,
        do_sample=True
    )

    # extract only the generated portion
    generated_ids = outputs[0][len(inputs.input_ids[0]):]
    answer = tokenizer_smol.decode(generated_ids, skip_special_tokens=True)

    return answer.strip(), actual_tokens

print("inference function defined")

inference function defined


In [25]:
print("\n" + "="*60)
print("testing smollm3 with varying context lengths")
print("comparing native long context vs semantic retrieval")
print("="*60 + "\n")

# testing 4k to 128k with proper yarn configuration
context_sizes = [4000, 8000, 16000, 32000, 64000, 128000]
scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

smollm_results = []

for max_tokens in context_sizes:
    print(f"\n{'='*50}")
    print(f"testing with {max_tokens:,} token context")
    print(f"{'='*50}")

    predictions = []
    times = []
    actual_tokens_used = []

    for i, (query, ref) in enumerate(zip(test_queries, reference_answers)):
        print(f"query {i+1}: {query}")

        start = time.time()
        answer, tokens = ask_smollm(long_context, query, max_context_tokens=max_tokens)
        elapsed = time.time() - start

        times.append(elapsed)
        actual_tokens_used.append(tokens)
        predictions.append(answer)

        print(f"  time: {elapsed:.2f}s")
        print(f"  tokens: {tokens:,}")
        print(f"  answer: {answer}")

    # calculate rouge-l scores
    rouge_scores = [
        scorer.score(ref, pred)['rougeL'].fmeasure
        for pred, ref in zip(predictions, reference_answers)
    ]

    avg_rouge = np.mean(rouge_scores)
    avg_time = np.mean(times)
    avg_tokens = np.mean(actual_tokens_used)

    result = {
        'method': f'SmolLM3 ({max_tokens//1000}k)',
        'context_size': max_tokens,
        'rouge_l': avg_rouge,
        'avg_time': avg_time,
        'avg_tokens': avg_tokens,
        'ram_gb': psutil.virtual_memory().used / 1e9
    }

    smollm_results.append(result)

    print(f"\nresults for {max_tokens//1000}k context:")
    print(f"  rouge-l: {avg_rouge:.3f}")
    print(f"  avg time: {avg_time:.2f}s")
    print(f"  avg tokens: {avg_tokens:,.0f}")
    print(f"  ram: {result['ram_gb']:.2f} gb")

print("\nsmollm3 testing complete\n")



testing smollm3 with varying context lengths
comparing native long context vs semantic retrieval


testing with 4,000 token context
query 1: What is the Telemachy?
  time: 9.57s
  tokens: 4,083
  answer: The Telemachy refers to the part of the "Odyssey" that focuses on the adventures and growth of Telemachus, Odysseus' son, as he seeks information about his father's disappearance and his own future. This part of the epic is written in the second person, addressing Telemachus directly, and it spans from Book i, lines 80-iv, to Book xxiv, lines 1-iv. It is distinct from the rest of the epic, which is from the perspective of Odysseus. The Telemachy is significant because it provides an opportunity for the reader to see the world through the eyes of a young man who is trying to find his place and identity, both
query 2: Who is Polyphemos?
  time: 9.64s
  tokens: 4,081
  answer: Polyphemos was a Cyclops, a one-eyed giant, who appears in Homer's "Odyssey." He is one of the monsters that Ody

In [26]:
def calculate_token_usage(method_name, context, queries,
                          chunk_size=500, overlap=100, top_k=5):
    """calculate total tokens processed by each method"""
    tokenizer_qwen = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-3B-Instruct")

    # encode with truncation to avoid warning
    full_context_tokens = len(tokenizer_qwen.encode(
        context,
        truncation=True,
        max_length=500000
    ))
    words = context.split()

    result = {
        'method': method_name,
        'full_doc_tokens': full_context_tokens,
        'tokens_per_query': [],
        'total_tokens': 0,
        'embedding_tokens': 0
    }

    if method_name == "Full Context (Hypothetical)":
        for query in queries:
            query_tokens = len(tokenizer_qwen.encode(query))
            tokens = full_context_tokens + query_tokens + 100
            result['tokens_per_query'].append(tokens)
        result['total_tokens'] = sum(result['tokens_per_query'])

    elif method_name == "Truncated Context":
        for query in queries:
            query_tokens = len(tokenizer_qwen.encode(query))
            tokens = 4000 + query_tokens + 100
            result['tokens_per_query'].append(tokens)
        result['total_tokens'] = sum(result['tokens_per_query'])

    elif method_name == "Naive Chunking":
        num_chunks = len(words) // chunk_size
        avg_chunk_tokens = full_context_tokens // num_chunks if num_chunks > 0 else full_context_tokens

        for query in queries:
            query_tokens = len(tokenizer_qwen.encode(query))
            tokens = (avg_chunk_tokens * 3) + query_tokens + 100
            result['tokens_per_query'].append(tokens)

        result['total_tokens'] = sum(result['tokens_per_query'])
        result['embedding_tokens'] = full_context_tokens

    elif method_name == "DolphinMind RAG":
        estimated_chunks = int((len(words) - chunk_size) / (chunk_size - overlap)) + 1
        avg_chunk_tokens = full_context_tokens // (len(words) // chunk_size)

        for query in queries:
            query_tokens = len(tokenizer_qwen.encode(query))
            tokens = (avg_chunk_tokens * top_k) + query_tokens + 100
            result['tokens_per_query'].append(tokens)

        result['total_tokens'] = sum(result['tokens_per_query'])
        result['embedding_tokens'] = full_context_tokens

    return result

print("token analysis function defined")

token analysis function defined


In [27]:
print("\n" + "="*60)
print("token efficiency analysis")
print("="*60 + "\n")

methods = [
    "Full Context (Hypothetical)",
    "Truncated Context",
    "Naive Chunking",
    "DolphinMind RAG"
]

token_results = []

print("calculating token usage for each method...\n")

for method in methods:
    result = calculate_token_usage(method, long_context, test_queries)
    token_results.append(result)

    print(f"{method}:")
    print(f"  total tokens processed: {result['total_tokens']:,}")
    print(f"  tokens per query: {result['total_tokens'] // len(test_queries):,}")
    if result['embedding_tokens'] > 0:
        print(f"  embedding overhead (one-time): {result['embedding_tokens']:,}")
    print()

print("token efficiency comparison:")
full_tokens = token_results[0]['total_tokens']
for result in token_results[1:]:
    reduction = ((full_tokens - result['total_tokens']) / full_tokens) * 100
    print(f"  {result['method']}: {reduction:.1f}% fewer tokens vs full context")

print("\ntoken analysis complete\n")



token efficiency analysis

calculating token usage for each method...

Full Context (Hypothetical):
  total tokens processed: 1,053,488
  tokens per query: 351,162

Truncated Context:
  total tokens processed: 12,326
  tokens per query: 4,108

Naive Chunking:
  total tokens processed: 6,419
  tokens per query: 2,139
  embedding overhead (one-time): 351,054

DolphinMind RAG:
  total tokens processed: 10,481
  tokens per query: 3,493
  embedding overhead (one-time): 351,054

token efficiency comparison:
  Truncated Context: 98.8% fewer tokens vs full context
  Naive Chunking: 99.4% fewer tokens vs full context
  DolphinMind RAG: 99.0% fewer tokens vs full context

token analysis complete



In [28]:
print("\n" + "="*60)
print("final comparison: all methods")
print("="*60 + "\n")

# results from previous notebooks
previous_results = [
    {'method': 'Truncated (Qwen)', 'rouge_l': 0.128, 'time': 14.63, 'tokens': 4100, 'approach': 'Baseline'},
    {'method': 'Naive Chunking (Qwen)', 'rouge_l': 0.132, 'time': 8.79, 'tokens': 2800, 'approach': 'TF-IDF Retrieval'},
    {'method': 'DolphinMind (Qwen)', 'rouge_l': 0.174, 'time': 58.67, 'tokens': 2500, 'approach': 'Semantic RAG'},
    {'method': 'RLM-Tools (Qwen)', 'rouge_l': 0.149, 'time': 15.34, 'tokens': 2100, 'approach': 'Tool Calling (RLM)'},
    {'method': 'Map-Reduce (Qwen)', 'rouge_l': 0.097, 'time': 129.10, 'tokens': 45000, 'approach': 'Summarization'},
    {'method': 'Hierarchical (Qwen)', 'rouge_l': 0.094, 'time': 164.00, 'tokens': 52000, 'approach': 'Hierarchical Sum.'}
]

all_results = previous_results.copy()

# add smollm3 results
for r in smollm_results:
    all_results.append({
        'method': r['method'],
        'rouge_l': r['rouge_l'],
        'time': r['avg_time'],
        'tokens': r['avg_tokens'],
        'approach': 'Native Long Context'
    })

# sort by rouge-l
all_results_sorted = sorted(all_results, key=lambda x: x['rouge_l'], reverse=True)

print(f"{'Method':<30} {'ROUGE-L':>10} {'Time(s)':>10} {'Tokens':>12} {'Approach':>20}")
print("="*85)

for r in all_results_sorted:
    print(f"{r['method']:<30} {r['rouge_l']:>10.3f} {r['time']:>10.2f} "
          f"{r['tokens']:>12,.0f} {r['approach']:>20}")

print("="*85)


final comparison: all methods

Method                            ROUGE-L    Time(s)       Tokens             Approach
DolphinMind (Qwen)                  0.174      58.67        2,500         Semantic RAG
RLM-Tools (Qwen)                    0.149      15.34        2,100   Tool Calling (RLM)
SmolLM3 (16k)                       0.144       9.37       16,085  Native Long Context
Naive Chunking (Qwen)               0.132       8.79        2,800     TF-IDF Retrieval
SmolLM3 (8k)                        0.131       9.64        8,085  Native Long Context
Truncated (Qwen)                    0.128      14.63        4,100             Baseline
SmolLM3 (128k)                      0.125      25.25      128,085  Native Long Context
SmolLM3 (32k)                       0.121      10.24       32,085  Native Long Context
SmolLM3 (4k)                        0.111       9.56        4,084  Native Long Context
SmolLM3 (64k)                       0.101      14.16       64,085  Native Long Context
Map-Reduce 

In [29]:
print("\n" + "="*60)
print("key findings")
print("="*60 + "\n")

best = all_results_sorted[0]
dolphin = [r for r in all_results if 'DolphinMind' in r['method']][0]
best_smol = max(smollm_results, key=lambda x: x['rouge_l'])

print("best overall:")
print(f"  {best['method']}: {best['rouge_l']:.3f}")
print()

print("dolphinmind rag:")
print(f"  rouge-l: {dolphin['rouge_l']:.3f}")
print(f"  tokens: {dolphin['tokens']:,.0f}")
print(f"  time: {dolphin['time']:.2f}s")
print()

print(f"best smollm3 ({best_smol['context_size']//1000}k):")
print(f"  rouge-l: {best_smol['rouge_l']:.3f}")
print(f"  tokens: {best_smol['avg_tokens']:,.0f}")
print(f"  time: {best_smol['avg_time']:.2f}s")
print()

accuracy_diff = ((dolphin['rouge_l'] - best_smol['rouge_l']) / best_smol['rouge_l']) * 100
token_reduction = ((best_smol['avg_tokens'] - dolphin['tokens']) / best_smol['avg_tokens'] * 100)

print(f"dolphinmind vs smollm3 ({best_smol['context_size']//1000}k):")
print(f"  accuracy: {accuracy_diff:+.1f}%")
print(f"  token reduction: {token_reduction:.1f}%")
print()

temp_tokenizer = AutoTokenizer.from_pretrained(model_name_smol)
full_doc_tokens = len(temp_tokenizer.encode(long_context, truncation=True, max_length=500000))

print("efficiency:")
print(f"  full doc: {full_doc_tokens:,} tokens")
print(f"  rag: {dolphin['tokens']:,.0f} tokens/query")
print(f"  reduction: {((full_doc_tokens - dolphin['tokens']) / full_doc_tokens * 100):.1f}%")



key findings

best overall:
  DolphinMind (Qwen): 0.174

dolphinmind rag:
  rouge-l: 0.174
  tokens: 2,500
  time: 58.67s

best smollm3 (16k):
  rouge-l: 0.144
  tokens: 16,085
  time: 9.37s

dolphinmind vs smollm3 (16k):
  accuracy: +20.5%
  token reduction: 84.5%

efficiency:
  full doc: 348,962 tokens
  rag: 2,500 tokens/query
  reduction: 99.3%


In [30]:
results_package = {
    'smollm3_results': smollm_results,
    'token_analysis': token_results,
    'final_comparison': all_results_sorted,
    'metadata': {
        'document_words': len(long_context.split()),
        'document_tokens': full_doc_tokens,
        'num_queries': len(test_queries),
        'best_method': best['method'],
        'best_rouge_l': best['rouge_l']
    }
}

with open('final_evaluation_results.json', 'w') as f:
    json.dump(results_package, f, indent=2)

print("\n" + "="*60)
print("evaluation complete - results saved")
print("="*60)


evaluation complete - results saved
